# 04. Explainable Anomaly Alerting with SHAP
## Fan Predictive Maintenance System

This notebook demonstrates how the system combines **Isolation Forest Anomaly Scoring**, **RUL Predictions**, and **SHAP (SHapley Additive exPlanations)** to generate technician-friendly, explainable maintenance alerts.

In [ ]:
import sys
import os
sys.path.append('../')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap

from src.data_preprocessing import DataPreprocessor
from src.feature_engineering import FeatureEngineer
from src.anomaly_detection import AnomalyDetector
from src.rul_estimation import RULEstimator
from src.explainability import SHAPExplainer
from src.alert_generation import AlertGenerator

print("Modules imported successfully.")

### 1. Load Data & Pretrained Models

In [ ]:
preprocessor = DataPreprocessor(config_path='../config/config.yaml')
preprocessor.load_data('../data/predictive_maintenance_dataset.csv')
baseline = preprocessor.extract_baseline()

engineer = FeatureEngineer(config_path='../config/config.yaml')
baseline_eng, feature_names = engineer.engineer_features(baseline)
X_baseline = baseline_eng[feature_names].values

data_eng, _ = engineer.engineer_features(preprocessor.data)
X_all = data_eng[feature_names].values

detector = AnomalyDetector(config_path='../config/config.yaml')
detector.load('../models/isolation_forest_model.pkl')

rul_model = RULEstimator(config_path='../config/config.yaml')
rul_model.load('../models/rul_estimator_model.pkl')

explainer = SHAPExplainer(detector.model, X_baseline, feature_names)
alert_gen = AlertGenerator(config_path='../config/config.yaml')
print("All models and explainers initialized.")

### 2. SHAP Global Feature Importance

In [ ]:
importance = explainer.get_feature_importance_summary(X_all[:200])
imp_df = pd.DataFrame(list(importance.items()), columns=['Feature', 'Importance']).sort_values('Importance', ascending=True)

plt.figure(figsize=(10, 5))
plt.barh(imp_df['Feature'], imp_df['Importance'], color='#3498db')
plt.title('Global Feature Importance for Anomaly Detection (Mean |SHAP Value|)')
plt.xlabel('Mean |SHAP Value|')
plt.tight_layout()
plt.show()

### 3. Case Studies: Local Anomaly Explanations

In [ ]:
y_labels = preprocessor.get_labels()
anomaly_idx = np.where(y_labels == 1)[0]

for i in [0, 5, 10]: # Pick 3 distinct anomaly examples
    sample_idx = anomaly_idx[i]
    sample_row = preprocessor.data.iloc[sample_idx]
    sample_vec = X_all[sample_idx]
    
    # Inference
    score = detector.predict_proba(sample_vec.reshape(1, -1))[0]
    rul = rul_model.predict(sample_vec.reshape(1, -1))[0]
    explanation = explainer.explain_prediction(sample_vec)
    
    feat_dict = {
        'TotalVibration': sample_row['TotalVibration'],
        'Temp': sample_row['Temp'],
        'Voltage': sample_row['Voltage'],
        'Current': sample_row['Current']
    }
    
    alert = alert_gen.generate_alert(feat_dict, score, rul, explanation)
    print(alert_gen.format_alert_for_display(alert))